# [Laminar] Counter Flow Flame 2D

## Preamble

In [1]:
# Standard Library
import sys
import os
from pathlib import Path
import foamnordic as fno
import numpy as np
import matplotlib.pyplot as plt
import onsaemiro as osm

### Directory & Path

In [2]:
# FoamNordic Project Directory
PROJECT_DIR = Path("/scratch/<allocation-account>/<user>")
CASE_TYPE = "laminar"
CASE_NAME = "counterFlowFlame2D"
BASE_DIR = PROJECT_DIR / "Codes" / "FoamNordic"
MAIN_DIR = BASE_DIR / "foamnordic_tutorials" / "combustion"
OF_SCRIPT_DIR = BASE_DIR / "openfoam_tutorials" / CASE_TYPE / CASE_NAME

# Output Directory
OUTPUT_DIR = MAIN_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### Configuration

In [ ]:
# HPC configuration
ACCOUNT = "<allocation-account>"
PARTITION = "small"
TIME = "00:15:00"
N_NODES = 1
N_TASKS = 1
CPUS_PER_TASK = 1
MEM_PER_CPU = "16G"

# FoamNordic Slurm configuration
of_scheduler = fno.Slurm.openfoam(
    nodes=N_NODES,
    ntasks=N_TASKS,
    cpus_per_task=CPUS_PER_TASK,
    mem_per_cpu=MEM_PER_CPU,
)

scheduler = fno.Slurm(
    account=ACCOUNT,
    partition=PARTITION,
    time=TIME,
    openfoam=of_scheduler,
)

In [4]:
# FoamNordic configuration
SEED = 42
key = fno.Random.key(seed=SEED, scope="global")

## Example - OpenFOAM on FoamNordic (Baseline)

### Case Definition

In [5]:
# Initialize the OpenFOAM case
case = fno.OpenFOAM.Case(
    name=CASE_NAME,
    case_dir=OF_SCRIPT_DIR,
    run_dir=OUTPUT_DIR,
    of_cmd="module load openfoam/2512",
    shell="bash",
    application="reactingFoam",
)

case.initialize(ranks=N_TASKS, mesh="blockMesh", validate_mesh=True);

### Submit Job

In [ ]:
# Connect the OpenFOAM case and SLURM scheduler (launch a longship instance)
longship = fno.Longship(case=case, scheduler=scheduler) 

# Set sail for the OpenFOAM case (submit the job to the HPC cluster)
run = longship.launch(start_timeout=900)

[FoamNordic] Preparing mesh with blockMesh: counterFlowFlame2D
[FoamNordic] Mesh is ready: counterFlowFlame2D
[FoamNordic] Sailing submitted with Job ID: 829546
[FoamNordic] Sailing has launched with Job ID: 829546
[FoamNordic] Sailing started at: 2026-08-24T08:49:12
[FoamNordic] Sailing in background: counterFlowFlame2D


In [7]:
# Wait for the job to complete (polling the job status)
result = run.stop(force=False, timeout=3600, progress=True)

In [8]:
# Summary of the job result
result.summary(style="compact");

Job ID,Name,Status,Partition,Node,Elapsed
829546,counterFlowFlame2D,succeeded,small,rc5134,00:04:09


### Postprocessing

In [9]:
# Postprocessing
post = result.postprocess

velocity = post.field("U", time_idx=-1)
pressure = post.field("p", time_idx=-1)

print("U shape:", velocity.shape)
print("p shape:", pressure.shape)

statistics = post.statistics(
    ["U", "p"],
    time_idx=-1,
    verbose=True,
)

U shape: (4000, 3)
p shape: (4000,)


Field,Min,Max,Mean,Std,RMS
U,3.386320e-03,2.249064e-01,1.042155e-01,3.692458e-02,1.105636e-01
p,1.000000e+05,1.000000e+05,1.000000e+05,0.000000e+00,1.000000e+05
